In [28]:
#!/usr/bin/env python3
import re
import glob
import os

# directory containing your uni-dock logs
log_dir = r"E:\Specificty_docking\run\results\logs"

# find all .log files in that directory
log_paths = glob.glob(os.path.join(log_dir, "*.log"))

total_ligands        = 0
processed_by_batches = 0
processed_by_poses   = 0
lowest_score         = float('inf')

for log_path in log_paths:
    with open(log_path, 'r') as f:
        for line in f:
            # 1) extract “Total ligands:” and sum across logs
            m_tot = re.search(r"Total ligands:\s*(\d+)", line)
            if m_tot:
                total_ligands += int(m_tot.group(1))

            # 2) sum “Batch N size:” across logs
            m_batch = re.match(r"^\s*Batch\s+\d+\s+size:\s*(\d+)", line)
            if m_batch:
                processed_by_batches += int(m_batch.group(1))

            # 3) match the pose-1 affinity line and update min score
            m_pose = re.match(r"^\s*1\s+([-\d\.]+)", line)
            if m_pose:
                processed_by_poses += 1
                try:
                    score = float(m_pose.group(1))
                    if score < lowest_score:
                        lowest_score = score
                except ValueError:
                    pass

# final summary
print(f"Logs processed:               {len(log_paths)}")
print(f"Count of pose-1 lines =       {processed_by_poses}")
print(f"Total peptides in all jobs:   {total_ligands}")


if lowest_score == float('inf'):
    print("No affinity scores found in the logs.")
else:
    print(f"Lowest affinity observed:     {lowest_score:.3f} kcal/mol")


Logs processed:               4
Count of pose-1 lines =       10541
Total peptides in all jobs:   15000
Lowest affinity observed:     -7.548 kcal/mol


In [13]:
import py3Dmol

def visualize_protein(
    pdb_path: str,
    center: tuple,
    size: tuple,
    surface_type: str = "SAS",
    surface_color: str = "white",
    surface_opacity: float = 0.6,
    cartoon_style: dict = {"color": "spectrum"},
    box_color: str = "deeppink",           # default changed to pink
    box_thickness: float = 0.2,        # thicker than before
    cross_fraction: float = 0.2,       # size of center cross relative to box
    cross_color: str = "yellow",
    cross_thickness: float = 0.15,
    arrow_scale: float = 0.15,
    arrow_label_size: int = 12
):
    """
    Load PDB, render smooth surface + cartoon,
    draw a thick pink grid‐box, yellow cross at center,
    annotated axes, and navy background.
    """
    # Load PDB
    with open(pdb_path, "r") as f:
        pdb_data = f.read()

    # Initialize viewer & background
    view = py3Dmol.view(width=800, height=600)
    view.setBackgroundColor('navy')
    view.addModel(pdb_data, "pdb")

    # Smooth surface
    surf = py3Dmol.SAS if surface_type.upper()=="SAS" else py3Dmol.VDW
    view.addSurface(surf,
                    {"opacity": surface_opacity, "color": surface_color},
                    {})

    # Cartoon
    view.setStyle({}, {"cartoon": cartoon_style})

    # Unpack box
    xc, yc, zc = center
    sx, sy, sz = size
    hx, hy, hz = sx/2, sy/2, sz/2

    # Corners & edges
    corners = [
        (xc-hx, yc-hy, zc-hz), (xc+hx, yc-hy, zc-hz),
        (xc+hx, yc+hy, zc-hz), (xc-hx, yc+hy, zc-hz),
        (xc-hx, yc-hy, zc+hz), (xc+hx, yc-hy, zc+hz),
        (xc+hx, yc+hy, zc+hz), (xc-hx, yc+hy, zc+hz),
    ]
    edges = [
        (0,1),(1,2),(2,3),(3,0),
        (4,5),(5,6),(6,7),(7,4),
        (0,4),(1,5),(2,6),(3,7),
    ]

    # Draw thick pink box
    for i, j in edges:
        a, b = corners[i], corners[j]
        view.addCylinder({
            "start": {"x": a[0], "y": a[1], "z": a[2]},
            "end":   {"x": b[0], "y": b[1], "z": b[2]},
            "radius": box_thickness,
            "color": box_color
        })

    # Draw a small yellow cross at box center
    cross_len = min(sx, sy, sz) * cross_fraction
    # X arm
    view.addCylinder({
        "start": {"x": xc - cross_len/2, "y": yc,      "z": zc},
        "end":   {"x": xc + cross_len/2, "y": yc,      "z": zc},
        "radius": cross_thickness,
        "color": cross_color
    })
    # Y arm
    view.addCylinder({
        "start": {"x": xc,      "y": yc - cross_len/2, "z": zc},
        "end":   {"x": xc,      "y": yc + cross_len/2, "z": zc},
        "radius": cross_thickness,
        "color": cross_color
    })

    # Annotated axes (unchanged)
    base = (xc + hx*0.8, yc - hy*0.8, zc - hz*0.8)
    arrow_len = min(sx, sy, sz) * arrow_scale
    arrow_rad = arrow_len * 0.05
    axes = [("X",(1,0,0),"red"), ("Y",(0,1,0),"green"), ("Z",(0,0,1),"blue")]

    for label, direction, color in axes:
        dx, dy, dz = direction
        view.addArrow({
            "start": {"x": base[0], "y": base[1], "z": base[2]},
            "end":   {"x": base[0]+dx*arrow_len,
                      "y": base[1]+dy*arrow_len,
                      "z": base[2]+dz*arrow_len},
            "radius": arrow_rad, "color": color
        })
        view.addLabel(label, {
            "position": {"x": base[0]+dx*arrow_len*1.1,
                         "y": base[1]+dy*arrow_len*1.1,
                         "z": base[2]+dz*arrow_len*1.1},
            "fontColor": color, "fontSize": arrow_label_size, "backgroundColor":"white"
        })

    # Finalize
    view.zoomTo()
    view.show()

In [26]:
#or VDW MS SES (no surface)
visualize_protein(
    pdb_path=r"e:\Specificty_docking\run\GOAT.pdb",
    center=(-3, 5, 0),
    size=(40, 40, 40),
    surface_type="MS", 
    surface_opacity=0.8,
    surface_color="lightgray"
)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [8]:
import py3Dmol
import ipywidgets as widgets
from IPython.display import display, clear_output

def interactive_box_view(
    pdb_path: str,
    init_center=(0, 0, 0),
    init_size=(30, 30, 30),
    surface_type="SAS",
    surface_opacity=0.8,
    surface_color="white",
    cartoon_style={"color": "spectrum"},
    box_color="deeppink",
    box_thickness=0.2
):
    """
    Interactive viewer: rebuilds the scene on each slider move to ensure 
    the grid updates live.
    """
    # Read PDB
    with open(pdb_path, "r") as f:
        pdb_data = f.read()

    # ----- NEW: parse all ATOM/HETATM coordinates to get true min/max -----
    coords = []
    for line in pdb_data.splitlines():
        if line.startswith(("ATOM", "HETATM")):
            # PDB fixed‐width columns: x=30-38, y=38-46, z=46-54
            x = float(line[30:38])
            y = float(line[38:46])
            z = float(line[46:54])
            coords.append((x, y, z))
    xs, ys, zs = zip(*coords)
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    z_min, z_max = min(zs), max(zs)
    # -----------------------------------------------------------------------

    # Output area for the 3Dmol view
    view_out = widgets.Output(layout={'border': '1px solid black'})

    # Slider definitions: use bounding‐box extents for X/Y/Z
    slider_x = widgets.FloatSlider(
        min=x_min, max=x_max,
        step=0.5, value=init_center[0], description='X'
    )
    slider_y = widgets.FloatSlider(
        min=y_min, max=y_max,
        step=0.5, value=init_center[1], description='Y'
    )
    slider_z = widgets.FloatSlider(
        min=z_min, max=z_max,
        step=0.5, value=init_center[2], description='Z'
    )
    # size sliders unchanged
    slider_sx = widgets.FloatSlider(min=1, max=init_size[0]*2, step=1, value=init_size[0], description='Sx')
    slider_sy = widgets.FloatSlider(min=1, max=init_size[1]*2, step=1, value=init_size[1], description='Sy')
    slider_sz = widgets.FloatSlider(min=1, max=init_size[2]*2, step=1, value=init_size[2], description='Sz')

    # Function to rebuild and display the view
    def draw(change=None):
        with view_out:
            clear_output(wait=True)
            v = py3Dmol.view(width=700, height=500)
            v.setBackgroundColor('navy')
            v.addModel(pdb_data, "pdb")
            surf = py3Dmol.SAS if surface_type.upper()=="SAS" else py3Dmol.VDW
            v.addSurface(surf, {"opacity": surface_opacity, "color": surface_color}, {})
            v.setStyle({}, {"cartoon": cartoon_style})

            xc, yc, zc = slider_x.value, slider_y.value, slider_z.value
            sx, sy, sz = slider_sx.value, slider_sy.value, slider_sz.value
            hx, hy, hz = sx/2, sy/2, sz/2

            corners = [
                (xc-hx, yc-hy, zc-hz), (xc+hx, yc-hy, zc-hz),
                (xc+hx, yc+hy, zc-hz), (xc-hx, yc+hy, zc-hz),
                (xc-hx, yc-hy, zc+hz), (xc+hx, yc-hy, zc+hz),
                (xc+hx, yc+hy, zc+hz), (xc-hx, yc+hy, zc+hz),
            ]
            edges = [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]
            for i, j in edges:
                a, b = corners[i], corners[j]
                v.addCylinder({
                    "start": {"x":a[0],"y":a[1],"z":a[2]},
                    "end":   {"x":b[0],"y":b[1],"z":b[2]},
                    "radius": box_thickness,
                    "color": box_color
                })

            v.zoomTo()
            display(v)

    for sl in (slider_x, slider_y, slider_z, slider_sx, slider_sy, slider_sz):
        sl.observe(draw, names='value')

    controls = widgets.VBox([widgets.HBox([slider_x, slider_y, slider_z]),
                             widgets.HBox([slider_sx, slider_sy, slider_sz])])
    display(controls, view_out)
    draw()

# Usage: call this once in your notebook
interactive_box_view(r"e:\Specificty_docking\run\GOAT.pdb")


Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

In [95]:
import pandas as pd

# 1. Load your data; if it's tab-delimited, set sep='\t'
input_path  = r'e:\Partners HealthCare Dropbox\Arvin Hajmirzaian\Cross_dock\all_sites.csv'     # ← replace with your actual file path
output_path = r'e:\Partners HealthCare Dropbox\Arvin Hajmirzaian\Cross_dock\all_sites_final.csv'    # ← desired path for the revised file

df = pd.read_csv(input_path)  # or pd.read_csv(input_path, sep='\t')

# 2. Count how many times each protein appears
protein_counts = df['protein'].value_counts()

# 3. Build new protein names
counters = {}
new_names = []
for name in df['protein']:
    if protein_counts[name] > 1:
        # this protein is duplicated → increment its counter
        counters[name] = counters.get(name, 0) + 1
        new_names.append(f"{name}_{counters[name]}")
    else:
        # unique → keep original
        new_names.append(name)

# 4. Replace the column
df['protein'] = new_names

# 5. Save to CSV
df.to_csv(output_path, index=False)

print(f"Revised CSV written to {output_path}")


Revised CSV written to e:\Partners HealthCare Dropbox\Arvin Hajmirzaian\Cross_dock\all_sites_final.csv


In [96]:
import os
import shutil
import pandas as pd

# — USER SETTINGS —————————————————————————————————————————————————————
csv_path       = r"e:\Partners HealthCare Dropbox\Arvin Hajmirzaian\Cross_dock\all_sites_final.csv"                   # your CSV with the 'protein' column
pdb_source_dir = r"e:\Partners HealthCare Dropbox\Arvin Hajmirzaian\Cross_dock\alphafold_pdb_files"  # folder containing e.g. A0A075B6H8.pdb
pdb_dest_dir   = r"e:\Partners HealthCare Dropbox\Arvin Hajmirzaian\Cross_dock\final_proteins_pdb"      # will be created to hold the renamed PDBs
# ————————————————————————————————————————————————————————————————————————

# 1. Load the CSV (comma-delimited by default)
df = pd.read_csv(csv_path, dtype=str)

# 2. Map base name → list of suffixed names
base_to_suffixes = {}
for prot in df["protein"]:
    parts = prot.rsplit("_", 1)
    if len(parts) == 2 and parts[1].isdigit():
        base = parts[0]
    else:
        base = prot
    base_to_suffixes.setdefault(base, []).append(prot)

# 3. Create destination folder (must not already exist)
if os.path.exists(pdb_dest_dir):
    raise FileExistsError(f"Destination folder '{pdb_dest_dir}' already exists. Remove it or choose another path.")
os.makedirs(pdb_dest_dir)

# 4. Copy & rename
for base, suffixed_list in base_to_suffixes.items():
    src = os.path.join(pdb_source_dir, f"{base}.pdb")
    if not os.path.isfile(src):
        print(f"Warning: '{base}.pdb' not found in source → skipping {len(suffixed_list)} entries")
        continue
    for newname in suffixed_list:
        dst = os.path.join(pdb_dest_dir, f"{newname}.pdb")
        shutil.copy2(src, dst)

print("Done – copied and renamed matching PDB files to:")
print(f"  {pdb_dest_dir}")


Done – copied and renamed matching PDB files to:
  e:\Partners HealthCare Dropbox\Arvin Hajmirzaian\Cross_dock\final_proteins_pdb


In [99]:
import os
import pandas as pd

# — USER SETTINGS ———————————————————————————————————————————————
csv_path    = r"e:\Partners HealthCare Dropbox\Arvin Hajmirzaian\Cross_dock\all_sites_final.csv"
pdb_dir     = r"e:\Partners HealthCare Dropbox\Arvin Hajmirzaian\Cross_dock\final_proteins_pdb"
output_path = csv_path   # overwrite; or set to a new path if you prefer
# ————————————————————————————————————————————————————————————————

# 1. Load CSV
df = pd.read_csv(csv_path, dtype=str)

# 2. Compute size_kb for each protein
sizes = []
for prot in df["protein"]:
    pdb_file = os.path.join(pdb_dir, f"{prot}.pdb")
    if os.path.isfile(pdb_file):
        size_bytes = os.path.getsize(pdb_file)
        size_kb    = round(size_bytes / 1024, 1)
    else:
        size_kb = None
    sizes.append(size_kb)

# 3. Add column and save
df["size_kb"] = sizes
df.to_csv(output_path, index=False)

print(f"Updated CSV written to {output_path}")


Updated CSV written to e:\Partners HealthCare Dropbox\Arvin Hajmirzaian\Cross_dock\all_sites_final.csv
